# Solution 6: Feature Extraction by Country - On-Demand Extremes DT

The On-Demand Extremes DT provides ultra-high-resolution data on a Lambert conformal grid. This solution uses Ireland as an example.

**Note**: The On-Demand DT uses longitudes in the range [0, 360], so we need to remap negative longitudes.

## Authentication

In [ ]:
%%capture cap
%run ../../desp-authentication.py

In [ ]:
output_1 = cap.stdout.split('}\n')
access_token = output_1[-1][0:-1]

## Imports

In [1]:
import earthkit.data
import earthkit.plots
from earthkit.geo import cartography
import os

## Live Request Toggle

In [2]:
LIVE_REQUEST = os.getenv("LIVE_REQUEST", "true").lower() == "true"
LIVE_REQUEST

True

## Select Your Country and Prepare Polygon

In [3]:
countries = ["Ireland"]

shapes = cartography.country_polygons(countries, resolution=500e5)

# Remap longitudes from [-180, 180] to [0, 360] (required for On-Demand DT)
for shape in shapes:
    for point in shape:
        lon = point[1]
        if lon < 0:
            point[1] = lon + 360

print(f"Prepared {len(shapes)} polygon(s) for {countries}")

Prepared 2 polygon(s) for ['Ireland']


## Build the Request

The `georef` geohash `gc7x9` covers Ireland.

In [ ]:
request = {
    "class": "d1",
    "dataset": "on-demand-extremes-dt",
    "expver": "0099",
    "stream": "oper",
    "date": "20250616",
    "time": 0,
    "type": "fc",
    "levtype": "sfc",
    "georef": "u4usq2", 
    "step": 12,
    "param": 167,
    "feature": {
        "type": "polygon",
        "shape": shapes,
    },
}

## Retrieve the Data

In [ ]:
if LIVE_REQUEST:
    data = earthkit.data.from_source(
        "polytope", "destination-earth", request,
        address="polytope.lumi.apps.dte.destination-earth.eu",
        stream=False,
    )
else:
    data = earthkit.data.from_source("file", "../../on-demand-extremes-dt/data/on-demand-fe-country-training.grib")

## Convert to xarray and Plot

In [ ]:
ds = data.to_xarray()
ds

In [ ]:
chart = earthkit.plots.Map(domain=["Ireland"])
chart.point_cloud(ds["2t"], units="celsius")
chart.coastlines()
chart.borders()
chart.gridlines()
chart.title("On-Demand Extremes DT - 2m Temperature for Ireland")
chart.legend()
chart.show()

## Bonus: Resolution Comparison

Notice how the On-Demand DT provides much higher spatial density compared to the Extremes DT (Exercise 5). The Lambert conformal grid gives sub-kilometer resolution in the focus area.